# Testing & Evaluating Agents

*Notebook 09*

An agent that works in a demo might fail on real inputs.

The only way to know if your agent is actually working — and to catch regressions after changes — is a structured evaluation process.
<br>
<br>

**Topics:**
- Why demos are not enough

- Building a golden test set — saving and versioning over time

- Pass/fail checks for factual correctness

- Checking output quality

- Rubric-based evaluation with a judge agent

- Comparing two agent versions systematically

- Regression checks after model swaps, prompt changes, or schema updates

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, function_tool
from typing import Annotated
from pydantic import BaseModel, Field

MODEL = "gpt-5-mini"
REASONING_MODEL = "gpt-5"



print("✅ Ready!")

---

## 🎯 The Problem

A demo is one carefully chosen input that works.

Evaluation is what separates a working agent from one that just looks like it works.

---

## 🤖 The Agent Under Test

We'll use a simple product assistant — simple enough to keep the evaluation framework in focus.

In [ ]:
PRODUCTS = {
    "P001": {"name": "Wireless Headphones", "price": 149.99, "stock": 23, "rating": 4.5},
    "P002": {"name": "Phone Case", "price": 19.99, "stock": 0, "rating": 4.1},
    "P003": {"name": "Laptop Stand", "price": 49.99, "stock": 7, "rating": 4.8},
    "P004": {"name": "USB-C Cable", "price": 12.99, "stock": 150, "rating": 3.9}
}


@function_tool
def get_product(product_id: str) -> str:
    """Get details for a product by ID."""
    try:
        p = PRODUCTS[product_id.upper()]
        status = "In stock" if p["stock"] > 0 else "Out of stock"
        return (
            f"{p['name']} | ${p['price']:.2f} | "
            f"Rating: {p['rating']}/5 | {status} ({p['stock']} units)"
        )
    except KeyError:
        return f"Product '{product_id}' not found."


@function_tool
def list_products() -> str:
    """List all available products."""
    lines = [f"{pid}: {p['name']} — ${p['price']:.2f}" for pid, p in PRODUCTS.items()]
    return "\n".join(lines)

product_instructions = (
    "You are a helpful product assistant.\n"
    "Answer questions about products using the available tools.\n"
    "Always look up specific product details before answering price or stock questions.\n"
    "If a product is out of stock, mention this clearly."
)

product_agent = Agent(
    name="ProductAssistant",
    instructions=product_instructions,
    model=MODEL,
    tools=[get_product, list_products]
)

# --------------------------------------------------------------
print("✅ Product assistant ready")
print(f"    Tools: get_product, list_products")
print(f"    Products in catalog: {len(PRODUCTS)}")

Before building test cases, decide what success looks like for your agent.

For a product assistant: factual accuracy on prices and stock, and clear responses.

Different agents need different success criteria — define yours before writing a single test case.

---

## 📋 Part 1: Building a Golden Test Set

A golden test set is a collection of inputs where you know what the correct output should look like.

Each test case has a question and one or more criteria the answer must satisfy.

A useful test set covers three things: happy paths, edge cases (missing items, ambiguous inputs), and known failure modes you've already seen the agent get wrong.

In [ ]:
# Each test case: question, expected facts that must appear in the answer
TEST_CASES = [
    {
        "id": "T01",
        "input": "How much does the Laptop Stand cost?",
        "must_contain": ["49.99"],
        "must_not_contain": ["149.99", "19.99"],
        "description": "Price lookup — correct product"
    },
    {
        "id": "T02",
        "input": "Is the Phone Case in stock?",
        "must_contain": ["out of stock", "not in stock", "unavailable", "0"],
        "must_not_contain": [],
        "description": "Stock status — out of stock item"
    },
    {
        "id": "T03",
        "input": "What products do you carry?",
        "must_contain": ["Wireless Headphones", "Laptop Stand"],
        "must_not_contain": [],
        "description": "Product listing — all items shown"
    },
    {
        "id": "T04",
        "input": "Tell me about product P999.",
        "must_contain": ["not found", "doesn't exist", "no product", "P999"],
        "must_not_contain": [],
        "description": "Missing product — graceful error handling"
    },
    {
        "id": "T05",
        "input": "Which product has the highest rating?",
        "must_contain": ["Laptop Stand", "4.8"],
        "must_not_contain": [],
        "description": "Comparative question — requires checking all products"
    },
]

# --------------------------------------------------------------
print(f"✅ Golden test set: {len(TEST_CASES)} test cases")
for tc in TEST_CASES:
    print(f"   {tc['id']}: {tc['description']}")

Test sets should be saved to a file and version-controlled alongside your agent code — that way regressions can be traced to specific instruction or model changes.

In [ ]:
# Save the test set to a file for version control
import json

with open("test_cases.json", "w") as f:
    json.dump(TEST_CASES, f, indent=2)

# Load it back in a future session
# TEST_CASES = json.load(open("test_cases.json"))

print("✅ Test cases saved to test_cases.json")

---

## ✅ Part 2: Pass/Fail Checks

The simplest evaluation: check whether required facts appear in the output.  

Fast, deterministic, and catches factual errors automatically.

In [ ]:
def check_output(output: str, must_contain: list[str], must_not_contain: list[str]) -> tuple[bool, list[str]]:
    """Check if output contains all required and none of the forbidden strings."""
    output_lower = output.lower()
    failures = []

    for s in must_contain:
        if s.lower() not in output_lower:
            failures.append(f"Missing required string: {s}")

    for s in must_not_contain:
        if s.lower() in output_lower:
            failures.append(f"Contains forbidden string: {s}")

    return not failures, failures

# --------------------------------------------------------------
print("✅ check_output() helper ready")

#### Run the Full Test Suite

In [ ]:
print("🧪 RUNNING TEST SUITE")
print("=" * 60)

results = []

for test_case in TEST_CASES:

    result = await Runner.run(product_agent, input=test_case["input"])

    output = result.final_output

    passed, failures = check_output(
        output,
        test_case["must_contain"],
        test_case["must_not_contain"]
    )

    results.append({"id": test_case["id"], "passed": passed, "failures": failures})

    icon = "✅" if passed else "❌"
    print(f"\n{icon} {test_case['id']}: {test_case['description']}")
    if not passed:
        for failure in failures:
            print(f"      → {failure}")
    print(f"   Output: {output}")

# Summary
passed_count = sum(1 for r in results if r["passed"])
print(f"\n{'='*60}")
print(f"📊 Results: {passed_count}/{len(results)} passed")
if passed_count == len(results):
    print("✅ All tests passed!")
else:
    print(f"❌ {len(results) - passed_count} test(s) failed — review output above")
print("=" * 60)

### 💡 Why This Works

Pass/fail checks are deterministic — the same test produces the same result every time.

They don't measure quality, just correctness on known facts.

## 🏆 Part 3: Rubric-Based Evaluation with a Judge Agent

Pass/fail checks can't measure quality — helpfulness, clarity, tone.

For that, use a judge agent: an LLM that scores the output against a rubric.

The judge agent has no built-in knowledge of your data — pass the ground truth into the prompt so it can verify factual accuracy instead of guessing.

For larger datasets, pass only the relevant slice (e.g., the entry for the product being asked about), not the entire catalog.

In [ ]:
class EvalResult(BaseModel):
    score: Annotated[int, Field(ge=1, le=5)]
    factual_accuracy: bool
    helpful: bool
    clear: bool
    feedback: str

judge_instructions = (
    "You evaluate product assistant responses.\n\n"
    "Scoring rubric (1-5):\n"
    "5 — Accurate, helpful, clear, directly answers the question\n"
    "4 — Accurate and helpful, minor clarity issues\n"
    "3 — Partially helpful, some inaccuracy or missing information\n"
    "2 — Mostly unhelpful or inaccurate\n"
    "1 — Wrong, misleading, or completely unhelpful\n\n"
    "Be strict. A score of 5 requires both correct facts AND a genuinely helpful response."
)

judge_agent = Agent(
    name="EvaluationJudge",
    instructions=judge_instructions,
    model=REASONING_MODEL,
    output_type=EvalResult
)

# --------------------------------------------------------------
print("✅ Judge agent ready")
print(f"    Model: {REASONING_MODEL} (stronger reasoning for evaluation)")

#### Run Rubric Evaluation

In [ ]:
# Select a few test cases for rubric evaluation
eval_cases = TEST_CASES[:3]

print("🏆 RUBRIC EVALUATION")
print("=" * 60)

eval_results = []
v1_passes = []

catalog = "\n".join(
    f"{pid}: {p['name']} — ${p['price']:.2f} — "
    f"{'In stock' if p['stock'] > 0 else 'Out of stock'} ({p['stock']} units) — Rating: {p['rating']}/5"
    for pid, p in PRODUCTS.items()
)

for test_case in eval_cases:
    # Get agent response

    agent_result = await Runner.run(product_agent, input=test_case["input"])

    agent_output = agent_result.final_output

    passed, _ = check_output(agent_output, test_case["must_contain"], test_case["must_not_contain"])
    v1_passes.append(passed)

    judge_input = (
        f"Product catalog (ground truth):\n{catalog}\n\n"
        f"Question: {test_case['input']}\n\n"
        f"Agent response: {agent_output}\n\n"
        f"Evaluate this response. Use the product catalog to verify factual accuracy."
    )

    judge_result = await Runner.run(judge_agent, input=judge_input)

    evaluation = judge_result.final_output
    eval_results.append(evaluation)

    print(f"\n📝 {test_case['id']}: {test_case['description']}")
    print(f"   Score:    {'⭐' * evaluation.score} ({evaluation.score}/5)")
    print(f"   Accurate: {'✅' if evaluation.factual_accuracy else '❌'}  "
          f"Helpful: {'✅' if evaluation.helpful else '❌'}  "
          f"Clear: {'✅' if evaluation.clear else '❌'}")
    print(f"   Feedback: {evaluation.feedback}")

avg_score = sum(e.score for e in eval_results) / len(eval_results)
v1_scores = [e.score for e in eval_results]
print(f"\n{'='*60}")
print(f"📊 Average score: {avg_score:.1f}/5")
print("=" * 60)

---

# Version A — current agent (defined above)
agent_v1 = product_agent

# v2 trades verbosity for structure: lead with the answer, suggest alternatives proactively, cap at 3 sentences.
# The test set will tell us whether the tighter format costs us factual accuracy.

# Version B — improved instructions
v2_instructions = (
    "You are a concise and accurate product assistant.\n"
    "When answering questions:\n"
    "- Always use get_product or list_products before answering\n"
    "- Lead with the direct answer, then supporting details\n"
    "- For out-of-stock items, proactively suggest alternatives if any are available\n"
    "- For missing products, confirm the ID and suggest listing all products\n"
    "Keep responses under 3 sentences."
)

agent_v2 = Agent(
    name="ProductAssistant_v2",
    instructions=v2_instructions,
    model=MODEL,
    tools=[get_product, list_products]
)

# --------------------------------------------------------------
print("✅ Both agent versions ready for comparison")

In [ ]:
# Version A — current agent (defined above)
agent_v1 = product_agent

# Version B — improved instructions
v2_instructions = (
    "You are a concise and accurate product assistant.\n"
    "When answering questions:\n"
    "- Always use get_product or list_products before answering\n"
    "- Lead with the direct answer, then supporting details\n"
    "- For out-of-stock items, proactively suggest alternatives if any are available\n"
    "- For missing products, confirm the ID and suggest listing all products\n"
    "Keep responses under 3 sentences."
)

agent_v2 = Agent(
    name="ProductAssistant_v2",
    instructions=v2_instructions,
    model=MODEL,
    tools=[get_product, list_products]
)

# --------------------------------------------------------------
print("✅ Both agent versions ready for comparison")

comparison_cases = TEST_CASES[:3]

print("🔄 VERSION COMPARISON")
print("=" * 60)

v1_scores = []
v2_scores = []
v1_passes = []
v2_passes = []

for test_case in comparison_cases:

    v1_result = await Runner.run(agent_v1, input=test_case["input"])

    v2_result = await Runner.run(agent_v2, input=test_case["input"])

    catalog = "\n".join(
        f"{pid}: {p['name']} — ${p['price']:.2f} — "
        f"{'In stock' if p['stock'] > 0 else 'Out of stock'} ({p['stock']} units) — Rating: {p['rating']}/5"
        for pid, p in PRODUCTS.items()
    )

    for label, result in [("v1", v1_result), ("v2", v2_result)]:
        output = result.final_output
        passed, _ = check_output(output, test_case["must_contain"], test_case["must_not_contain"])
        judge_input = (
            f"Product catalog (ground truth):\n{catalog}\n\n"
            f"Question: {test_case['input']}\n\n"
            f"Agent response: {output}\n\n"
            f"Evaluate this response. Use the product catalog to verify factual accuracy."
        )

        judge_result = await Runner.run(judge_agent, input=judge_input)

        score = judge_result.final_output.score
        if label == "v1":
            v1_scores.append(score)
            v1_passes.append(passed)
        else:
            v2_scores.append(score)
            v2_passes.append(passed)

    print(f"\n{test_case['id']}: {test_case['description']}")
    print(f"   v1: {'✅' if v1_passes[-1] else '❌'} pass/fail  |  {'⭐' * v1_scores[-1]} ({v1_scores[-1]}/5)")
    print(f"   v2: {'✅' if v2_passes[-1] else '❌'} pass/fail  |  {'⭐' * v2_scores[-1]} ({v2_scores[-1]}/5)")

avg_v1 = sum(v1_scores) / len(v1_scores)
avg_v2 = sum(v2_scores) / len(v2_scores)
v1_pass_rate = sum(v1_passes) / len(v1_passes)
v2_pass_rate = sum(v2_passes) / len(v2_passes)
winner = "v2" if avg_v2 > avg_v1 else "v1" if avg_v1 > avg_v2 else "tie"

print(f"\n{'='*60}")
print(f"📊 Pass rate  — v1: {v1_pass_rate:.0%}  |  v2: {v2_pass_rate:.0%}")
print(f"📊 Avg score  — v1: {avg_v1:.1f}/5  |  v2: {avg_v2:.1f}/5")
print(f"🏆 Winner (by judge score): {winner}")
print("💡 Pass rate catches factual breakage; avg score catches quality drift. Don't ship a version that loses on one metric to win on the other.")
print(f"💡 Note: this comparison uses 3 test cases — real decisions need a larger test set.")
print("=" * 60)

In [ ]:
# Reuse v1 baseline from Part 3 — only run and evaluate v2
comparison_cases = TEST_CASES[:3]

print("🔄 VERSION COMPARISON")
print("=" * 60)

v2_scores = []
v2_passes = []

for i, test_case in enumerate(comparison_cases):

    v2_result = await Runner.run(agent_v2, input=test_case["input"])

    v2_output = v2_result.final_output
    passed, _ = check_output(v2_output, test_case["must_contain"], test_case["must_not_contain"])
    v2_passes.append(passed)

    judge_input = (
        f"Product catalog (ground truth):\n{catalog}\n\n"
        f"Question: {test_case['input']}\n\n"
        f"Agent response: {v2_output}\n\n"
        f"Evaluate this response. Use the product catalog to verify factual accuracy."
    )

    judge_result = await Runner.run(judge_agent, input=judge_input)

    v2_scores.append(judge_result.final_output.score)

    print(f"\n{test_case['id']}: {test_case['description']}")
    print(f"   v1: {'✅' if v1_passes[i] else '❌'} pass/fail  |  {'⭐' * v1_scores[i]} ({v1_scores[i]}/5)")
    print(f"   v2: {'✅' if v2_passes[-1] else '❌'} pass/fail  |  {'⭐' * v2_scores[-1]} ({v2_scores[-1]}/5)")

avg_v1 = sum(v1_scores) / len(v1_scores)
avg_v2 = sum(v2_scores) / len(v2_scores)
v1_pass_rate = sum(v1_passes) / len(v1_passes)
v2_pass_rate = sum(v2_passes) / len(v2_passes)
winner = "v2" if avg_v2 > avg_v1 else "v1" if avg_v1 > avg_v2 else "tie"

print(f"\n{'='*60}")
print(f"📊 Pass rate  — v1: {v1_pass_rate:.0%}  |  v2: {v2_pass_rate:.0%}")
print(f"📊 Avg score  — v1: {avg_v1:.1f}/5  |  v2: {avg_v2:.1f}/5")
print(f"🏆 Winner (by judge score): {winner}")
print("💡 Pass rate catches factual breakage; avg score catches quality drift. Don't ship a version that loses on one metric to win on the other.")
print(f"💡 Note: this comparison uses 3 test cases — real decisions need a larger test set.")
print("=" * 60)

---

## 💪 Practice Exercises

### Exercise 1: Extend the Test Set

*Covers: test case design — expanding evaluation coverage*

Add three more test cases to `TEST_CASES` that cover scenarios not currently tested, then run the full suite.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Extend the Test Set
# --------------------------------------------------------------
# Objective: Add edge cases the current test set doesn't cover.

# TODO 1: Think of 3 scenarios the current tests don't cover.
#          Ideas: asking about price of cheapest item, asking in a different
#          phrasing, asking about a real product by name instead of ID,
#          multi-part questions, ambiguous queries.

# TODO 2: Add 3 new test cases to TEST_CASES following the same format:
#          {"id": ..., "input": ..., "must_contain": [...],
#           "must_not_contain": [...], "description": ...}

# TODO 3: Re-run the test suite on all cases (original + new)
#          Print a summary showing which tests pass and which fail

# --- Write your code below this line ---

### Exercise 2: Regression Check

*Covers: regression testing — catching instruction degradation*

Change the agent's instructions in a way that might break something, then run the test suite to catch the regression.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Regression Check
# --------------------------------------------------------------
# Objective: Use the test suite to catch a deliberate regression.

# TODO 1: Create a degraded agent version with instructions that are
#          intentionally weaker — for example:
#          - "Answer questions about products. Be brief."
#          (No instruction to look up products, no error handling guidance)

# TODO 2: Run the full TEST_CASES suite against the degraded agent

# TODO 3: Compare pass/fail counts between the original agent
#          and the degraded version

# TODO 4: Print which specific tests regressed (passed before, fail now)
#          This is what a regression check looks like in practice

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Build a golden test set early:**

- Include happy paths, edge cases, and known failure modes

- Each test case needs an input and verifiable expected facts

- Aim for 10–20 cases that cover your most important scenarios

- Save to a file and version-control alongside agent code — regressions trace to specific changes
<br>
<br>

**Pass/fail checks for factual correctness:**

- `must_contain` and `must_not_contain` strings catch factual errors reliably

- Fast, deterministic, and easy to extend

- Good for prices, names, status values, and other verifiable facts
<br>
<br>

**Judge agents for quality:**

- Use `REASONING_MODEL` and structured output (`output_type=EvalResult`)

- Define a clear rubric — vague criteria produce inconsistent scores

- Use numeric scores so results are comparable across runs
<br>
<br>

**Always compare versions on the same test set:**

- Never change instructions without running evals before and after

- Use your test sets to confirm upgrades actually improve quality

- A score improvement on some tests with regressions on others is a net loss

- When an eval fails and you can't tell why from the output, tracing shows the full tool call sequence — covered in Lesson 25: Tracing & Observability

---

## 📍 Next Step

**Notebook 10: Web Search**  

Give your agent access to live information from the web using OpenAI's built-in web search tool.

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-09-testing-and-evaluating-agents)

---